## Data Migration: SQL to Postgres

In [107]:
import os
import pandas as pd
import pyodbc
pyodbc.pooling = False
import psycopg2
from psycopg2.extras import execute_values
from dotenv import load_dotenv


## 1. LOAD CREDENTIALS

In [108]:
load_dotenv()

True

In [109]:
sql_host = os.getenv("SQLSERVER_HOST")
sql_db = os.getenv("SQLSERVER_DB")
sql_user = os.getenv("SQLSERVER_USER")
sql_password = os.getenv("SQLSERVER_PASSWORD")


In [110]:
print(f"SQL SERVER HOST:{sql_host}") 
print(f"SQL DB:{sql_db}") 
print(f"SQL USER:{sql_user}") 
print(f"SQL PASSWORD:{sql_password}") 

SQL SERVER HOST:DESKTOP-P6OS4A2\SQLEXPRESS
SQL DB:AdventureWorks2022
SQL USER:loadingusr
SQL PASSWORD:Toluca2000


In [111]:
pg_host= os.getenv("PostgreSQL_HOST")
pg_port= os.getenv("PostgreSQL_PORT")
pg_db= os.getenv("PostgreSQL_DB")
pg_user= os.getenv("PostgreSQL_USER")
pg_password= os.getenv("PostgreSQL_PASSWORD")

In [112]:
print(f"POSTGRE HOST:{pg_host}")
print(f"POSTGRE PORT:{pg_port}")
print(f"POSTGRE DB:{pg_db}")
print(f"POSTGRE USER:{pg_user}")
print(f"POSTGRE PASSWORD:{pg_password}")

POSTGRE HOST:localhost
POSTGRE PORT:5432
POSTGRE DB:AdventureWorks2022
POSTGRE USER:loadingusr
POSTGRE PASSWORD:Toluca2000


## Connect to SQL Server

In [113]:
print("connect to SQL Server...")
print(f"    Server:{sql_host}")
print(f"    User:{sql_user}")
print(f"    Password:{sql_password}")
print(f"    DB:{sql_db}")

connect to SQL Server...
    Server:DESKTOP-P6OS4A2\SQLEXPRESS
    User:loadingusr
    Password:Toluca2000
    DB:AdventureWorks2022


In [114]:
print(pyodbc.drivers())

['SQL Server', 'SQL Server Native Client RDA 11.0', 'ODBC Driver 17 for SQL Server']


In [115]:
try:
    sql_conn_string =(
        f"USER={sql_user};"
        f"PASSWORD={sql_password};"
        f"SERVER={sql_host};"
        f"DATABASE={sql_db};"
        f"DRIVER={{SQL Server}};"
        f"Trusted_Conection=yes;"
        f"MARS_Connection=yes;"
        f"MultipleActiveResultSets=True;"
    )
    sql_conn = pyodbc.connect(sql_conn_string)
    sql_cursor = sql_conn.cursor()
    print(f"[SUCCESS]Conection to SQL Server completed!")
    

except Exception as e:
    print(f"SQL Server connection failed:{e}")
    print(""" How to Toubleshoot:
         >1. Check server name in .env is correct
         >2. Verify SQL Server is runing
         >3. Check Windows authentication is enable
          ... 
""")

[SUCCESS]Conection to SQL Server completed!


## Connect to PostgreSQL

In [116]:
print("connect to SQL PostgreSQL...")
print(f"    Server:{pg_host}")
print(f"    User:{pg_db}")


connect to SQL PostgreSQL...
    Server:localhost
    User:AdventureWorks2022


In [117]:
try:
    pg_conn = psycopg2.connect(
        host=pg_host,
        port=pg_port,
        database=pg_db,
        user=pg_user,
        password=pg_password
    )

    pg_cursor=pg_conn.cursor()
    pg_cursor.execute("SELECT version();")
    pg_version = pg_cursor.fetchone()[0]
    print("Conected to PostgreSQL")
    print(f"    Version: {pg_version[:50]}...\n")

except psycopg2.OperationalError as e:
    print(f"Postgres Connection failed{e}")
    print(""" How to trobleshoot:
            > 1. Check Postgres is running 
            > 2. Verify user name + password
            > 3. Check db exists 
          
          ...
""")
    
except Exception as e:
    print(f"Unexpected error: {e}")
    raise

Conected to PostgreSQL
    Version: PostgreSQL 18.3 on x86_64-windows, compiled by msv...



## 4. Define the tables to migrate

### Migration Orden
- Sales.Customer (Dependencies)
- Production.Product (Dependencies)
- Production.ProductCategory (No Dependencies)
- Purchasing.ProductVendor (Depedencies)
- Person.Adress (No Dependencies)


In [118]:
#tables_to_migrate= ['Sales.Customer', 'Production.ProductCategory', 'Production.Product', 'Purchasing.ProductVendor', 'Person.Address']
#print(tables_to_migrate)

#tables_to_migrate2= ['Customer', 'ProductCategory', 'Product', 'ProductVendor', 'Address']
#print(tables_to_migrate2)

tables_to_migrate= ['Production.Product']
#print(tables_to_migrate)

tables_to_migrate2= ['Product']
#print(tables_to_migrate2)



In [119]:
print("Tables to migrate:")
for i, table in enumerate(tables_to_migrate2, 1):
    print(f"    {i}.    {table}")

total_no_tbls = len(tables_to_migrate)
print(f"\nTotal number of tables to migrate: {total_no_tbls}")

   

Tables to migrate:
    1.    Product

Total number of tables to migrate: 1


## 5. Run pre migration checks

In [120]:
print("==" * 50)
print(">> Rows counts")
print("==" * 50)

>> Rows counts


In [121]:
test_query = "SELECT COUNT(*) AS total_rows FROM Production.Product;"
sql_cursor.execute(test_query)
count = sql_cursor.fetchone()[0]

print(f"Result: {count}")

Result: 504


In [122]:
baseline_counts = {}

try:
    for table in tables_to_migrate:
        
        row_count_query = f"SELECT COUNT(*) as total_rows FROM {table}"
        sql_cursor.execute(row_count_query)
        count = sql_cursor.fetchone()[0]

        baseline_counts[table] = count
        print(f"{table:25} {count:>20} rows")

    total_rows = sum(baseline_counts.values())
    print(f"{'-' *30}")
    print(f"{'TOTAL':25} {total_rows:>20,} rows ")     
    print(f"Baseline captured!")
    
    sql_cursor.close()
    sql_conn.close() 
          
        
except Exception as e:
    print (f"Failed to get baseline counts: {e}")
    raise

Production.Product                         504 rows
------------------------------
TOTAL                                      504 rows 
Baseline captured!


In [123]:
print("==" * 50)
print(">> Check 2: NULL COUNTS (Purchasing.ProductVendor )")
print("==" * 50)

>> Check 2: NULL COUNTS (Purchasing.ProductVendor )


In [124]:
try:
    sql_conn_string =(
        f"USER={sql_user};"
        f"PASSWORD={sql_password};"
        f"SERVER={sql_host};"
        f"DATABASE={sql_db};"
        f"DRIVER={{SQL Server}};"
        f"Trusted_Conection=yes;"
        f"MARS_Connection=yes;"
        f"MultipleActiveResultSets=True;"
    )
    sql_conn = pyodbc.connect(sql_conn_string)
    sql_cursor = sql_conn.cursor()
    print(f"[SUCCESS]Conection to SQL Server completed!")
    

except Exception as e:
    print(f"SQL Server connection failed:{e}")
    print(""" How to Toubleshoot:
         >1. Check server name in .env is correct
         >2. Verify SQL Server is runing
         >3. Check Windows authentication is enable
          ... 
""")

quality_issues= []

try:
    print(">> Check 2: NULL COUNTS (Purchasing.ProductVendor)\n")
    sql_cursor.execute("""SELECT COUNT(*) AS null_count 
                       FROM Purchasing.ProductVendor WHERE OnOrderQty IS NULL""")
    null_OnQty= sql_cursor.fetchone()[0]
    if null_OnQty >0:
        quality_issues.append(f"  {null_OnQty} customer with NULL On Order Qty ")
    #print(quality_issues)

    print(">> Check 3: MIN ORDER QUANTITY ...")
    sql_cursor.execute("""SELECT COUNT(*) as Minimal_quantity_product  
                          FROM Purchasing.ProductVendor 
                          WHERE MinOrderQty <=1 """)
    
    Minimal_quantity_product = sql_cursor.fetchone()[0]
    if Minimal_quantity_product > 0:
        quality_issues.append(f"   - {Minimal_quantity_product:,} with only 1 product ...")
    print(quality_issues)

except Exception as e:
    print (f"Failed to get baseline counts: {e}")
    raise

print("\nCHECK 4:INVALID ADDRESS WITH OUT NUMBER")
sql_cursor.execute(""" SELECT COUNT(*) AS INVALID_ADRESS 
                        FROM Person.Address
                        WHERE AddressLine1 IS NOT NULL""")

sql_cursor.close()
sql_conn.close() 

[SUCCESS]Conection to SQL Server completed!
>> Check 2: NULL COUNTS (Purchasing.ProductVendor)

>> Check 3: MIN ORDER QUANTITY ...
['  305 customer with NULL On Order Qty ', '   - 285 with only 1 product ...']

CHECK 4:INVALID ADDRESS WITH OUT NUMBER


In [125]:
conn =  pyodbc.connect(
        f"USER={sql_user};"
        f"PASSWORD={sql_password};"
        f"SERVER={sql_host};"
        f"DATABASE={sql_db};"
        f"DRIVER={{SQL Server}};"
        f"Trusted_Conection=yes;"
    )

query= 'SELECT AddressLine1 FROM Person.Address'
df = pd.read_sql_query(query,conn)
str_sql= df
df_address=pd.DataFrame(str_sql)
df_address


C:\Users\IT\AppData\Local\Temp\ipykernel_26684\1354227241.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query,conn)


,AddressLine1
0,#500-75 O'Connor Street
1,#9900 2700 Production Way
2,"00, rue Saint-Lazare"
3,"02, place de Fontenoy"
4,"035, boulevard du Montparnasse"
...,...
19609,Zur Lindung 7
19610,Zur Lindung 7
19611,Zur Lindung 764
19612,Zur Lindung 78


In [126]:
data_list = []
def counter(df_address):
            c = 0          
            for i in df_address['AddressLine1']:
                eval = any(char.isdigit() for char in i)
                if eval is False:
                    #print("Detecting Invalid adress...")
                    data_list.append({'Wrong_AddressLine1':i})                   
                    c += 1
            return c
                    
cad = df_address
print(f"Number of registers without number in address: {counter(cad)}")
pd_invalid=pd.DataFrame(data_list)
pd_invalid

Number of registers without number in address: 207


,Wrong_AddressLine1
0,Adirondack Factory Outlet
1,Ames Plaza
2,Amity Plaza
3,Arcadia Crossing
4,Attaché de Presse
...,...
202,Wharfdale Road
203,White Mountain Mall
204,Woodbury Commons
205,Wrentham Village


In [127]:
## CHECK 5: GROUPING CUSTOMER PERSON BY CITY

## 6. Get table Schema

In [128]:
print("=" * 65)
print("ANALIZE TABLE SCHEMA")
print("=" * 65)

ANALIZE TABLE SCHEMA


In [129]:
table_schema={}

try:
    for table in tables_to_migrate2:
        schema_query= f"""
         SELECT 
            COLUMN_NAME, 
            DATA_TYPE, 
            CHARACTER_MAXIMUM_LENGTH,
            IS_NULLABLE 
        FROM INFORMATION_SCHEMA.COLUMNS
        WHERE table_name = '{table}'
        ORDER BY
            ORDINAL_POSITION 
"""
        schema_df = pd.read_sql(schema_query,conn)
        #print(schema_df)
        table_schema[table]=schema_df

        print(f"\n{table}")
        print("-" * 10)
        print(schema_df)
        table_schema[table] = schema_df
        print("\n\n")
                
except Exception as e:
    pass



Product
----------
              COLUMN_NAME         DATA_TYPE  CHARACTER_MAXIMUM_LENGTH  \
0               ProductID               int                       NaN   
1                    Name          nvarchar                      50.0   
2           ProductNumber          nvarchar                      25.0   
3                MakeFlag               bit                       NaN   
4       FinishedGoodsFlag               bit                       NaN   
5                   Color          nvarchar                      15.0   
6        SafetyStockLevel          smallint                       NaN   
7            ReorderPoint          smallint                       NaN   
8            StandardCost             money                       NaN   
9               ListPrice             money                       NaN   
10                   Size          nvarchar                       5.0   
11    SizeUnitMeasureCode             nchar                       3.0   
12  WeightUnitMeasureCode      

C:\Users\IT\AppData\Local\Temp\ipykernel_26684\421717186.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  schema_df = pd.read_sql(schema_query,conn)


## 7. DEFINE DATA TYPE MAPPINGS

In [130]:
print("=" * 65)
print("DATA TYPE MAPPING")
print("=" * 65)

DATA TYPE MAPPING


In [131]:
type_mapping = {
   'int': 'INTEGER',
    'bigint': 'BIGINT',
    'smallint': 'SMALLINT',
    'tinyint': 'SMALLINT',
    'bit': 'BOOLEAN',
    'decimal': 'NUMERIC',
    'numeric': 'NUMERIC',
    'money': 'NUMERIC(19,4)',
    'smallmoney': 'NUMERIC(10,4)',
    'float': 'DOUBLE PRECISION',
    'real': 'REAL',
    'datetime': 'TIMESTAMP',
    'datetime2': 'TIMESTAMP',
    'smalldatetime': 'TIMESTAMP',
    'date': 'DATE',
    'time': 'TIME',
    'char': 'VARCHAR',
    'varchar': 'VARCHAR',
    'nchar': 'VARCHAR',
    'nvarchar': 'VARCHAR',
    'text': 'TEXT',
    'ntext': 'TEXT',
    'NaN': 'NULL',
    'NaT':'NULL',
        
}

In [132]:
print("SQL Server to PostgreSQL type mapping ")
print ()
for sql_type, pg_type in list(type_mapping.items()):
    print(f"        {sql_type:15} --->       {pg_type}")

SQL Server to PostgreSQL type mapping 

        int             --->       INTEGER
        bigint          --->       BIGINT
        smallint        --->       SMALLINT
        tinyint         --->       SMALLINT
        bit             --->       BOOLEAN
        decimal         --->       NUMERIC
        numeric         --->       NUMERIC
        money           --->       NUMERIC(19,4)
        smallmoney      --->       NUMERIC(10,4)
        float           --->       DOUBLE PRECISION
        real            --->       REAL
        datetime        --->       TIMESTAMP
        datetime2       --->       TIMESTAMP
        smalldatetime   --->       TIMESTAMP
        date            --->       DATE
        time            --->       TIME
        char            --->       VARCHAR
        varchar         --->       VARCHAR
        nchar           --->       VARCHAR
        nvarchar        --->       VARCHAR
        text            --->       TEXT
        ntext           --->       TEXT
 

In [133]:
## 8. CREATE TABLES IN POSTGRES

In [134]:
print("=" * 65)
print("CREATE TABLES IN POSTGRES")
print("=" * 65)

CREATE TABLES IN POSTGRES


In [135]:
try:
    for table in tables_to_migrate2:
        schema = table_schema[table]
        pg_table = table.lower()
        pg_cursor.execute(f"DROP TABLE IF EXISTS {pg_table} CASCADE")
        column_definitions = []
        for inx, row in schema.iterrows():
            col_name = row['COLUMN_NAME'].lower()
            sql_type = row['DATA_TYPE']

            base_type = sql_type.lower()
            pg_type = type_mapping.get(base_type, 'TEXT')

            condition_1 = inx == 0                  #Must be first column in the table
            condition_2 = col_name.endswith('id')   #Must start with ID
            condition_3 = 'int' in sql_type.lower() #Must be INT data type

            if condition_1 and condition_2 and condition_3:
                column_definitions.append(f"{col_name} SERIAL PRIMARY KEY")
            else:
                column_definitions.append(f"{col_name} {pg_type}")

        column_string= ",\n                ".join(column_definitions)
        create_query = f"""
        CREATE TABLE {pg_table} (
            {column_string}
            
        )
        """
        pg_cursor.execute(create_query)
        pg_conn.commit()
    
    print("\n + " + "=" * 55)
    print("[SUCESS] ---> All tables created sucessfuly!")

except psycopg2.Error as e:
    print(f"Postgres experienced an error while creating a table: {e}")
    pg_conn.rollback()
    raise 
          
except Exception as e:
    print(f"Unexpected issue: {e}")



 + =======================================================
[SUCESS] ---> All tables created sucessfuly!


## 9. TEST MIGRATION WITH ONE TABLE

In [136]:
print("=" * 50)
print("Testing migration with single table")
print("=" * 50)

Testing migration with single table


In [137]:
test_table = 'Product'
test_table2 = 'Production.Product'
pg_table = test_table.lower()
print(pg_table)
print(test_table2)
print(test_table)


product
Production.Product
Product


In [138]:
try:
    sql_conn_string =(
        f"USER={sql_user};"
        f"PASSWORD={sql_password};"
        f"SERVER={sql_host};"
        f"DATABASE={sql_db};"
        f"DRIVER={{SQL Server}};"
        f"Trusted_Conection=yes;"
        f"MARS_Connection=yes;"
        f"MultipleActiveResultSets=True;"
    )
    sql_conn = pyodbc.connect(sql_conn_string)
    sql_cursor = sql_conn.cursor()
    print(f"[SUCCESS]Conection to SQL Server completed!")
    

except Exception as e:
    print(f"SQL Server connection failed:{e}")
    print(""" How to Toubleshoot:
         >1. Check server name in .env is correct
         >2. Verify SQL Server is runing
         >3. Check Windows authentication is enable
          ... 
""")

[SUCCESS]Conection to SQL Server completed!


In [139]:
#funcion PYTHON corregida por CLAUDE IA
try:
    #1. EXTRACT
    extract_query = f"SELECT * FROM {test_table2}"
    print(f"1. Extracting data: {extract_query}")
    test_work = pd.read_sql(extract_query, sql_conn)
    print(f"   Read {len(test_work):,} rows")

    # 2. TRANSFORM
    print("2. Transforming data types ...")
    test_df = test_work.copy()

    # ✅ Limpieza correcta: NaT / NaN / None reales → None
    for col in test_df.columns:
        if pd.api.types.is_datetime64_any_dtype(test_df[col]):
            # Timestamps: NaT → None
            test_df[col] = test_df[col].where(test_df[col].notna(), other=None)

        elif pd.api.types.is_float_dtype(test_df[col]):
            # Floats que son realmente enteros con nulos
            if test_df[col].dropna().apply(lambda x: x == int(x)).all():
                test_df[col] = test_df[col].astype(object).where(test_df[col].notna(), other=None)
            else:
                test_df[col] = test_df[col].where(test_df[col].notna(), other=None)

        elif pd.api.types.is_integer_dtype(test_df[col]):
            # Enteros con nullable int (Int64) → object con None
            test_df[col] = test_df[col].astype(object).where(test_df[col].notna(), other=None)

        elif pd.api.types.is_object_dtype(test_df[col]):
            # Strings vacíos y residuos textuales
            test_df[col] = test_df[col].replace({"": None, "NaT": None, "NaN": None, 'NaT': None})

           

    if 'FinishedGoodsFlag' in test_df.columns:
        test_df['FinishedGoodsFlag'] = test_df['FinishedGoodsFlag'].astype('bool')
        print("   [SUCCESS] ---> Converted FinishedGoodsFlag: BIT ---> BOOLEAN")

    if 'MakeFlag' in test_df.columns:
        test_df['MakeFlag'] = test_df['MakeFlag'].astype('bool')
        print("   [SUCCESS] ---> Converted MakeFlag: BIT ---> BOOLEAN")

    # 3. PREPARE
    print("3. Preparing data for loading ...")
    columns = [col.lower() for col in test_df.columns]
    column_string = ', '.join(columns)

    # ✅ itertuples preserva None correctamente, to_numpy() puede romperlo
    data_tuples = [
        tuple(None if (v is not None and isinstance(v, float) and pd.isna(v)) else v for v in row)
        for row in test_df.itertuples(index=False, name=None)
    ]

    insert_query = f"""
        INSERT INTO {pg_table} ({column_string})
        VALUES %s
    """
    print(f"   Table:   {pg_table}")
    print(f"   Columns: {column_string}")
    #print(f"   Data:    {data_tuples}")
    print(f"   Rows:    {len(data_tuples):,}")

    # 4. LOAD
    print("4. Inserting into PostgreSQL ...")
    execute_values(pg_cursor, insert_query, data_tuples, page_size=1000)
    pg_conn.commit()
    print(f"   Loaded {len(data_tuples):,} rows")

    # 5. VERIFY
    print("5. Verifying ...")
    pg_cursor.execute(f"SELECT COUNT(*) FROM {pg_table}")
    pg_count = pg_cursor.fetchone()[0]
    sql_count = baseline_counts[test_table2]

    if pg_count == sql_count:
        print(f"   [SUCCESS] ---> {pg_count:,} == {sql_count:,}")
    else:
        print(f"   [FAILED]  ---> {pg_count:,} != {sql_count:,}")

    print(f"\n✅ {test_table} migration test completed!")

except Exception as e:
    pg_conn.rollback()
    print(f"\n❌ Error en tabla {test_table}: {e}")
    raise


1. Extracting data: SELECT * FROM Production.Product


   Read 504 rows
2. Transforming data types ...
   [SUCCESS] ---> Converted FinishedGoodsFlag: BIT ---> BOOLEAN
   [SUCCESS] ---> Converted MakeFlag: BIT ---> BOOLEAN
3. Preparing data for loading ...
   Table:   product
   Columns: productid, name, productnumber, makeflag, finishedgoodsflag, color, safetystocklevel, reorderpoint, standardcost, listprice, size, sizeunitmeasurecode, weightunitmeasurecode, weight, daystomanufacture, productline, class, style, productsubcategoryid, productmodelid, sellstartdate, sellenddate, discontinueddate, rowguid, modifieddate
   Rows:    504
4. Inserting into PostgreSQL ...
   Loaded 504 rows
5. Verifying ...
   [SUCCESS] ---> 504 == 504

✅ Product migration test completed!


C:\Users\IT\AppData\Local\Temp\ipykernel_26684\1423310563.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  test_work = pd.read_sql(extract_query, sql_conn)
